In [ ]:
import numpy as np
import ufl

from mpi4py import MPI

import dolfinx
import dolfinx.fem.petsc
import dolfinx.mesh
import basix.ufl
from petsc4py import PETSc


length, height = 1.0, 1.0
Nx, Ny = 5,5
domain = dolfinx.mesh.create_rectangle(
    MPI.COMM_WORLD,
    [np.array([-1., -1.]), np.array([length, height])],
    [Nx, Ny],
    cell_type=dolfinx.mesh.CellType.triangle,
)
# domain = new_domain[0]

def up_bottom_boundary(x):
    #on_left = np.isclose(x[0], knots1[0])
    on_bottom = np.isclose(x[1], -1.)
    #on_right = np.isclose(x[0], knots1[-1])
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top

def all_around_boundary(x):
    on_left = np.isclose(x[0], -1.)
    on_bottom = np.isclose(x[1], 1.)
    on_right = np.isclose(x[0], length)
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top|on_left|on_right

dim = domain.topology.dim
print(f"Mesh topology dimension d={dim}.")

degree = 2
#shape = (dim,)  # this means we want a vector field of size `dim`
v_elem = basix.ufl.element(
    "Lagrange", 
    domain.topology.cell_name(), 
    degree=degree,
)
V = dolfinx.fem.functionspace(domain, v_elem)

u_sol = dolfinx.fem.Function(V, name="solution")

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

x = ufl.SpatialCoordinate(domain)

f = dolfinx.fem.Constant(domain, 0.1)
a_term = 0.1 + ufl.exp(3.*(x[0]+x[1]))

custom_metadata = {"quadrature_degree": 10}
custom_dx = ufl.Measure("dx", domain=domain, metadata=custom_metadata)
a = ufl.inner(a_term* ufl.grad(u), ufl.grad(v)) * custom_dx
l = ufl.inner(f,v)*custom_dx


def left(x):
    return np.isclose(x[0], -1.)
def right(x):
    return np.isclose(x[0], length)
def up(x):
    return np.isclose(x[1], height)
def bottom(x):
    return np.isclose(x[1], -1.)


left_dofs = dolfinx.fem.locate_dofs_geometrical(V, left)
right_dofs = dolfinx.fem.locate_dofs_geometrical(V, right)
up_dofs = dolfinx.fem.locate_dofs_geometrical(V, up)
down_dofs = dolfinx.fem.locate_dofs_geometrical(V, bottom)
zero_vec = PETSc.ScalarType(0)
bcs = [
    dolfinx.fem.dirichletbc(zero_vec, left_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, right_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, up_dofs, V),
    dolfinx.fem.dirichletbc(zero_vec, down_dofs, V),
]
problem = dolfinx.fem.petsc.LinearProblem(
    a, l, u=u_sol, bcs=bcs,
    petsc_options_prefix="poisson",
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps"}
)
problem.solve()

In [ ]:
import dolfinx.plot
import pyvista

# 1. Create a "Nodal" DG space of the same degree for plotting
# By default, DG with no variant specified uses Lagrange (nodal)
v_plot_elt = basix.ufl.element(
    "DG", 
    "triangle", 
    degree=2
)
V_plot = dolfinx.fem.functionspace(domain, v_plot_elt)

# 2. Interpolate your computed solution (u_dg) into the nodal space
#u_plot = dolfinx.fem.Function(V_plot)
#error_ufl = ufl.ln(ufl.sqrt((u_dg-f)**2)+1e-13)
error_ufl = u_sol
error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
u_error = dolfinx.fem.Function(V_plot)
u_error.interpolate(error_expr)

# 3. Now use V_plot for the VTK mesh generation
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
grid.point_data["u"] = u_error.x.array.real
grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
plotter = pyvista.Plotter()
grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
plotter.view_xy()
plotter.show(jupyter_backend="static")

Dual

In [ ]:
interpol_elem = basix.ufl.element(
    "Lagrange", 
    domain.topology.cell_name(), 
    degree=degree+1,
)
interpol_V = dolfinx.fem.functionspace(domain, interpol_elem)

u_higher = dolfinx.fem.Function(interpol_V)
u_higher.interpolate(u_sol)

# e_func = u_higher-u_sol
# norm_e_sq = dolfinx.fem.assemble_scalar(dolfinx.fem.form(ufl.inner(e_func, e_func) * custom_dx))
# norm_e = np.sqrt(domain.comm.allreduce(norm_e_sq, op=MPI.SUM))

# if norm_e > 1e-12:
#     rhs_dual = (1.0 / norm_e) * e_func
# else:
#     rhs_dual = dolfinx.fem.Constant(domain, zero_vec)
rhs_dual = dolfinx.fem.Constant(domain, 1.0) # for example
sigma = 0.05

#rhs_dual = ufl.exp(-((x[0] + 0.75)**2 + (x[1] + 0.75)**2) / sigma**2)

z = ufl.TrialFunction(interpol_V)
v_dual = ufl.TestFunction(interpol_V)
z_sol = dolfinx.fem.Function(interpol_V, name="dual_solution")

a_dual = ufl.inner(a_term* ufl.grad(z), ufl.grad(v_dual)) * custom_dx
l_dual = ufl.inner(rhs_dual,v_dual)*custom_dx

left_dofs_dual = dolfinx.fem.locate_dofs_geometrical(interpol_V, left)
right_dofs_dual = dolfinx.fem.locate_dofs_geometrical(interpol_V, right)
up_dofs_dual = dolfinx.fem.locate_dofs_geometrical(interpol_V, up)
down_dofs_dual = dolfinx.fem.locate_dofs_geometrical(interpol_V, bottom)

bcs_dual = [
    dolfinx.fem.dirichletbc(zero_vec, left_dofs_dual, interpol_V),
    dolfinx.fem.dirichletbc(zero_vec, right_dofs_dual, interpol_V),
    dolfinx.fem.dirichletbc(zero_vec, up_dofs_dual, interpol_V),
    dolfinx.fem.dirichletbc(zero_vec, down_dofs_dual, interpol_V),
]

problem_dual = dolfinx.fem.petsc.LinearProblem(
    a_dual, l_dual, u=z_sol, bcs=bcs_dual,
    petsc_options_prefix="dual_poisson",
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu", 
        "pc_factor_mat_solver_type": "mumps"}
)
problem_dual.solve()

In [ ]:
error_ufl = z_sol
error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
u_error = dolfinx.fem.Function(V_plot)
u_error.interpolate(error_expr)

# 3. Now use V_plot for the VTK mesh generation
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
grid.point_data["u"] = u_error.x.array.real
grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
plotter = pyvista.Plotter()
grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
plotter.view_xy()
plotter.show(jupyter_backend="static")
#plotter.show()

In [ ]:
# Define a piecewise-constant space for cell-wise error indicators
DG0 = dolfinx.fem.functionspace(domain, ("DG", 0))
v_dg = ufl.TestFunction(DG0)

# --- 1. Compute Cell Residual Integral: ||R||_K^2 ---
# Note: we use u_sol, not the TrialFunction 'u'
R_expr = f + ufl.div(a_term * ufl.grad(u_sol))
form_R2 = (R_expr**2) * v_dg * custom_dx
vec_R2 = dolfinx.fem.assemble_vector(dolfinx.fem.form(form_R2))

# --- 2. Compute Facet Residual Integral: ||r||_{partial K}^2 ---
n = ufl.FacetNormal(domain)
# ufl.jump(v, n) computes exactly [v] \cdot n
flux_jump = ufl.jump(a_term * ufl.grad(u_sol), n)
r_val = 0.5 * flux_jump

# We accumulate the facet integral into both adjacent cells by using v_dg('+') + v_dg('-')
# ufl.dS integrates only over interior facets, whereas ufl.ds integrates 
# only over exterior facets.
# v_dg('+') represents the test function restricted to the "positive" neighbour cell
# by having v_dg('+') + v_dg('-'), we add (r_val**2), the jump penalty is shared equally
# and added to the error indicators of both neighbouring cells that share that edge.
custom_dS = ufl.Measure("dS", domain=domain, metadata={"quadrature_degree": 10})
form_r2 = (r_val**2) * (v_dg('+') + v_dg('-')) * custom_dS
vec_r2 = dolfinx.fem.assemble_vector(dolfinx.fem.form(form_r2))

# --- 3. Compute Cell Diameters (h_K) ---
h_expr = dolfinx.fem.Expression(ufl.CellDiameter(domain), DG0.element.interpolation_points)
h_K = dolfinx.fem.Function(DG0)
h_K.interpolate(h_expr)

# --- 4. Compute Dual Hessian Integral: ||\nabla^2 z_h||_K^2 ---
# hessian_z = ufl.grad(ufl.grad(z_sol))
# form_omega2 = ufl.inner(hessian_z, hessian_z) * v_dg * custom_dx
# vec_omega2 = dolfinx.fem.assemble_vector(dolfinx.fem.form(form_omega2))
i_h_z_h = dolfinx.fem.Function(V)
i_h_z_h.interpolate(z_sol)

# --- 5. Assemble final quantities ---
# Note: In standard serial runs, the arrays map 1:1 to the cells.
h_array = h_K.x.array

# rho_K = ||R||_K + h_K^{-1/2} ||r||_{partial K}
# Since we assembled the squares, we take the square roots of the arrays
rho_K = np.sqrt(vec_R2.array) + (1.0 / np.sqrt(h_array)) * np.sqrt(vec_r2.array)

# omega_K = ||\nabla^2 z_h||_K
# omega_K = np.sqrt(vec_omega2.array)

# eta_K = h_K^2 * rho_K * omega_K
# eta_K = (h_array**4) * rho_K**2# * omega_K
dual_weight = z_sol - i_h_z_h
facet_form = (
    r_val * dual_weight('+') * v_dg('+')
    + r_val * dual_weight('-') * v_dg('-')
) * custom_dS
eta_form = R_expr*dual_weight*v_dg*custom_dx - facet_form
eta_vec = dolfinx.fem.assemble_vector(dolfinx.fem.form(eta_form))
eta_vec.scatter_forward()
eta_K = eta_vec.array
eta_K_abs = np.abs(eta_K)

In [ ]:
# --- 1. Dörfler Marking ---
theta = 0.3 

# Sort cell indices by error in descending order
sorted_indices = np.argsort(eta_K_abs)[::-1]
total_error = np.sum(eta_K_abs)

accumulated_error = 0.0
marked_cells = []

for idx in sorted_indices:
    marked_cells.append(idx)
    accumulated_error += eta_K_abs[idx]
    if accumulated_error >= theta * total_error:
        break

marked_cells = np.array(marked_cells, dtype=np.int32)
print(f"Marked {len(marked_cells)} out of {domain.topology.index_map(dim).size_local} cells for refinement.")

# --- 2. Extract Edges for FEniCSx ---
# Ensure 1D entities (edges) and connectivity are initialized
domain.topology.create_entities(1)
domain.topology.create_connectivity(dim, 1)
cell_to_edge = domain.topology.connectivity(dim, 1)

# Gather all edges associated with the marked cells
marked_edges = []
for cell in marked_cells:
    edges = cell_to_edge.links(cell)
    marked_edges.extend(edges)

# Remove duplicates to avoid feeding the same edge twice
marked_edges = np.unique(marked_edges).astype(np.int32)

# --- 3. Perform Refinement ---
# This generates a completely new mesh object
new_domain = dolfinx.mesh.refine(domain, marked_edges)

In [ ]:
error = np.sum(eta_K)
print(f"{error:.2e}")

## B-Splines

In [ ]:
from thbsplines.hierarchical_space import HierarchicalSpace
import numpy as np
import dolfinx
import pyvista
import basix.ufl
import ufl
from thbsplines.refinement import refine
from thbsplines.fenicsx.mesh import build_mesh
from thbsplines.fenicsx.functionspace import build_dofmap, fill_function_space, create_spline_space
from thbsplines.fenicsx.solvers import solve_problem, enforce_dirichlet_boundary
from thbsplines.fenicsx.adaptivity import dorfler_marking
from thbsplines.fenicsx.kernels import make_linear_kernel, make_bilinear_kernel
from thbsplines.fenicsx.postprocessing import map_spline_to_legendre
from thbsplines.fenicsx.forms import mark_cells, make_bilinear_form, make_linear_form

p0 = 1
L = 1.
h=1.
n_refinements = 1
knotsx = np.array([-1., (L-1.)/2., L], dtype=np.float64)
knotsx = refine(knotsx, p=p0, n_times=n_refinements)
#log_initial_mesh_size = np.log2(np.max(np.diff(knotsx)))
knotsy = refine(np.array([-1., 0., h], dtype=np.float64), p=p0, n_times=n_refinements)
err_cells = {}
hs = HierarchicalSpace(knots=[knotsx, knotsy], degrees=[p0])

p0_dual = p0+1
knotsx_dual = refine(knotsx, p0_dual, 0)
knotsy_dual = refine(knotsy, p0_dual, 0)
hs_dual = HierarchicalSpace([knotsx_dual, knotsy_dual], p0_dual)

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=False, buffer_zone_size=max(p0, p0_dual))
    hs_dual.refine(cells, level, refine_neighbours=False, buffer_zone_size=max(p0, p0_dual))
hs.hmesh.plot_cells()

mesh, thb_operators, N_max, _ = build_mesh(hs=hs)

**Problem definition**

In [ ]:
def all_around_boundaries(x):
    on_left = np.isclose(x[0], -1.)
    on_bottom = np.isclose(x[1], -1.)
    on_right = np.isclose(x[0], length)
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top|on_left|on_right

legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(mesh, legendre_elt)
custom_metadata = {"quadrature_degree": 4}
dx_custom = ufl.Measure("dx", domain=mesh, metadata=custom_metadata)

u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
x = ufl.SpatialCoordinate(mesh)
f = dolfinx.fem.Constant(domain=mesh, c=0.1)
a_term = 0.1 + ufl.exp(3.*(x[0]+x[1]))

a = ufl.inner(a_term* ufl.grad(u), ufl.grad(v)) * dx_custom
L_cell = ufl.inner(f, v) * dx_custom 

In [ ]:
dof_map, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=mesh, N_max=N_max, morton=True)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=mesh, N_max=N_max, thb_operators=thb_operators)
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=mesh,
                               N_max=N_max)
local_dofs = (hs.degrees[0]+1)**2
tabulate_A = make_bilinear_kernel(mesh, a, padded_dofs=N_max, local_dofs=local_dofs)
tabulate_L_cell = make_linear_kernel(mesh, L_cell, padded_dofs=N_max, local_dofs=local_dofs)

cell_domain = mark_cells(mesh=mesh)
a_cond = make_bilinear_form(mesh=mesh,
                            ufl_form=a,
                            trial_space=V_spline, test_space=V_spline,
                            coefficients=C_func, 
                            integrals=[(cell_domain, tabulate_A)])
l_cond = make_linear_form(
    mesh=mesh,
    ufl_form=L_cell,
    test_space=V_spline,
    coefficients=C_func,
    integrals=[(cell_domain, tabulate_L_cell)])

In [ ]:
forbidden_indices = enforce_dirichlet_boundary(hs, dof_map, bottom=True, right=True, top=True, left=True)
x_vec = solve_problem(hs=hs, a=a_cond, rhs=l_cond, dirichlet_indices=forbidden_indices,
                    dummy_index = np.max(padded_cells_to_dofs), V_spline=V_spline,
                    iterative=False, return_A=False)
u_dg = map_spline_to_legendre(hs, V, C_func, N_max, mesh, padded_cells_to_dofs, x_vec)

u_dg.x.scatter_forward()

In [ ]:
import dolfinx.plot
import pyvista

# 1. Create a "Nodal" DG space of the same degree for plotting
# By default, DG with no variant specified uses Lagrange (nodal)
v_plot_elt = basix.ufl.element(
    "DG", 
    "quadrilateral", 
    degree=p0+2
)
V_plot = dolfinx.fem.functionspace(mesh, v_plot_elt)

# 2. Interpolate your computed solution (u_dg) into the nodal space
#u_plot = dolfinx.fem.Function(V_plot)
#error_ufl = ufl.ln(ufl.sqrt((u_dg-f)**2)+1e-13)
error_ufl = u_dg
error_expr = dolfinx.fem.Expression(error_ufl, V_plot.element.interpolation_points)
u_error = dolfinx.fem.Function(V_plot)
u_error.interpolate(error_expr)

# 3. Now use V_plot for the VTK mesh generation
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
grid.point_data["u"] = u_error.x.array.real
grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
plotter = pyvista.Plotter()
grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
plotter.view_xy()
plotter.show(jupyter_backend="static")
#plotter.show()

# Adjoint problem

In [ ]:
def all_around_boundaries(x):
    on_left = np.isclose(x[0], -1.)
    on_bottom = np.isclose(x[1], -1.)
    on_right = np.isclose(x[0], length)
    on_top = np.isclose(x[1], height)
    return on_bottom|on_top|on_left|on_right

legendre_elt_dual = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0_dual,
    lagrange_variant=basix.LagrangeVariant.legendre
)
V_dual = dolfinx.fem.functionspace(mesh, legendre_elt_dual)
custom_metadata = {"quadrature_degree": 4}
dx_custom = ufl.Measure("dx", domain=mesh, metadata=custom_metadata)

z,v_dual = ufl.TrialFunction(V_dual), ufl.TestFunction(V_dual)

x = ufl.SpatialCoordinate(mesh)
sigma = 0.05
#rhs_dual = ufl.exp(-((x[0] + 0.75)**2 + (x[1] + 0.75)**2) / sigma**2)
rhs_dual = dolfinx.fem.Constant(mesh, 1.)

a_dual = ufl.inner(a_term* ufl.grad(z), ufl.grad(v_dual)) * dx_custom
L_cell_dual = ufl.inner(rhs_dual, v_dual) * dx_custom 

In [ ]:
_, thb_operators_dual, N_max_dual, _ = build_mesh(hs=hs_dual)
dof_map_dual, padded_cells_to_dofs_dual = build_dofmap(hierarchical_space=hs_dual, mesh=mesh, N_max=N_max_dual, morton=False)
C_func_dual, C_space_dual = fill_function_space(hierachical_space=hs_dual, mesh=mesh, N_max=N_max_dual, thb_operators=thb_operators_dual)
V_spline_dual = create_spline_space(cells_to_dofs=padded_cells_to_dofs_dual, mesh=mesh,
                               N_max=N_max_dual)
local_dofs_dual = (p0_dual+1)**2
tabulate_A_dual = make_bilinear_kernel(mesh, a_dual, padded_dofs=N_max_dual, local_dofs=local_dofs_dual)
tabulate_L_cell_dual = make_linear_kernel(mesh, L_cell_dual, padded_dofs=N_max_dual, local_dofs=local_dofs_dual)

a_cond_dual = make_bilinear_form(mesh=mesh,
                            ufl_form=a_dual,
                            trial_space=V_spline_dual, test_space=V_spline_dual,
                            coefficients=C_func_dual, 
                            integrals=[(cell_domain, tabulate_A_dual)])
l_cond_dual = make_linear_form(
    mesh=mesh,
    ufl_form=L_cell_dual,
    test_space=V_spline_dual,
    coefficients=C_func_dual,
    integrals=[(cell_domain, tabulate_L_cell_dual)])

In [ ]:
forbidden_indices_dual = enforce_dirichlet_boundary(hs_dual, dof_map_dual, bottom=True, right=True, top=True, left=True)
z_vec = solve_problem(hs=hs_dual, a=a_cond_dual, rhs=l_cond_dual, 
    dirichlet_indices=forbidden_indices_dual, dummy_index=np.max(padded_cells_to_dofs_dual), 
    V_spline=V_spline_dual, return_A=False)
z_dg = map_spline_to_legendre(hs_dual, V_dual, C_func_dual, N_max_dual, mesh, padded_cells_to_dofs_dual, z_vec)
z_dg.x.scatter_forward()

In [ ]:
v_plot_elt_dual = basix.ufl.element(
    "DG", 
    "quadrilateral", 
    degree=p0_dual+2
)
V_plot_dual = dolfinx.fem.functionspace(mesh, v_plot_elt_dual)
error_dual = z_dg
error_expr_dual = dolfinx.fem.Expression(error_dual, V_plot_dual.element.interpolation_points)
u_error_dual = dolfinx.fem.Function(V_plot_dual)
u_error_dual.interpolate(error_expr_dual)

# 3. Now use V_plot for the VTK mesh generation
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(V_plot_dual)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Attach the interpolated values
grid.point_data["u"] = u_error_dual.x.array.real
grid.set_active_scalars("u")

# 5. Plotting (with a 'shrink' to see your disconnected mesh boundaries!)
plotter = pyvista.Plotter()
grid_shrink = grid.shrink(0.95) # This makes the "disconnected" nature visible
plotter.add_mesh(grid_shrink, show_edges=False, cmap="turbo")
plotter.view_xy()
plotter.show(jupyter_backend="static")

In [ ]:
DG0 = dolfinx.fem.functionspace(mesh, ("DG", 0))
v_dg = ufl.TestFunction(DG0)

R_expr = f + ufl.div(a_term * ufl.grad(u_dg))

n = ufl.FacetNormal(mesh)
flux_jump = ufl.jump(a_term * ufl.grad(u_dg), n)
r_val = 0.5 * flux_jump
custom_dS = ufl.Measure("dS", domain=mesh, metadata={"quadrature_degree": 5})

i_h_z_h = dolfinx.fem.Function(V)
i_h_z_h.interpolate(z_dg)

dual_weight = z_dg - i_h_z_h
facet_form = (
    r_val * dual_weight('+') * v_dg('+')
    + r_val * dual_weight('-') * v_dg('-')
) * custom_dS
eta_form = R_expr*dual_weight*v_dg*dx_custom - facet_form
err_cells = dorfler_marking(hs, theta=0.3, local_error_form=dolfinx.fem.form(eta_form))